# Stable NanoGPT-1GPU Recipe + SSONN/MLP Comparison

This notebook combines the stable parts of `Deveraux-Parker/nanoGPT_1GPU_SPEEDRUN` with SSONN and MLP comparison experiments. It has three modes:

1. `RUN_MODE = "pretrain"`: train an **attention-only** GPT from scratch on FineWeb-Edu and save a checkpoint. This path is kept unchanged as the attention-only baseline.
2. `RUN_MODE = "ssonn"`: train GPT + residual SSONN MLP from scratch. SSONN cycles use local token schedules; after expansion the whole model is reinitialized and the train stream is reset so each SSONN iteration sees the same token budget/order.
3. `RUN_MODE = "mlp"`: train GPT + speedrun-style MLP from scratch for a dense MLP baseline.

The original speedrun uses nightly PyTorch, FlexAttention, BF16, FP8, and an RTX 4090. This notebook intentionally keeps its reproducible training recipe but uses stable PyTorch 2.6 compatible operations: FP16 autocast + GradScaler where stable and standard causal scaled-dot-product attention. Therefore it preserves the architecture scale, data format, optimizer idea, progressive context schedule, and update count, but does **not** claim the 90-minute/3.25-loss result on a TITAN RTX.



In [2]:
# One additional dependency is needed for GPT-2 BPE tokenization:
# %pip install tiktoken

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import glob
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tqdm.auto import tqdm

import ssonn.train.train as ssonn_train
from ssonn.metrics.edge_finder import EdgeFinder
from ssonn.metrics.nonlinearity_metrics import AbsGradientEdgeMetric
from ssonn.model.model import ScaledNet

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open the notebook from the repository root or experiments directory.")

DATA_DIR = REPO_ROOT / "data" / "finewebedu10B"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("This experiment is configured for CUDA training.")

# TITAN RTX has Tensor Cores but not BF16/FP8 training support required by the speedrun.
AMP_DTYPE = torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(torch.cuda.get_device_name(), "|", torch.cuda.get_device_properties(0).total_memory // 2**30, "GiB")

## Recipe Configuration

The model dimensions, data/tokenizer, Muon+Adam split, logit softcap, zero-initialized attention output projection, and `1750` optimizer updates are retained from the 4090 recipe.

The speedrun processes 32k-token local-attention sequences using FlexAttention. Standard PyTorch causal attention cannot safely run that on a 24GB TITAN RTX, so the stable version uses the same progressive attention window as its actual sequence length. Consequently it runs the same number of optimizer updates, but not the original 0.92B token budget.

In [ ]:
RUN_MODE = "ssonn"  # "pretrain", "ssonn", or "mlp"
SEED = 1337
USE_COMPILE = False  # Stable PyTorch compile is optional; start with False on TITAN RTX.

# FineWeb-Edu data produced by the speedrun repository uses GPT-2 BPE uint16 shards.
# Both the speedrun downloader and local tokenizer use these wildcard patterns.
TRAIN_GLOB = str(DATA_DIR / "*train_*.bin")
VAL_GLOB = str(DATA_DIR / "*val_*.bin")

# Recommended: download the speedrun's already GPT-2-tokenized 100M-token shards.
DOWNLOAD_PRETOKENIZED_DATA = False
PRETOKENIZED_TRAIN_SHARDS = 10  # 10 * 100M = about 1B training tokens.

# Slower fallback: tokenize the raw FineWeb-Edu stream locally.
PREPARE_DATA = False
PREPARE_TRAIN_TOKENS = 1_000_000_000
PREPARE_VAL_TOKENS = 10_000_000

@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 6
    n_embd: int = 768
    max_seq_len: int = 1792

MODEL_CONFIG = GPTConfig()

# Same update count, effective number of sequences/update, and context curriculum as the speedrun.
NUM_UPDATES = 50_000
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
WINDOW_START = 256
WINDOW_END = 1792
COOLDOWN_UPDATES = 640
EVAL_INTERVAL = 100
EVAL_BATCHES = 16
CHECKPOINT_INTERVAL = 250

# Conservative FP16 rates for TITAN RTX; the 4090 BF16 speedrun rates are unstable here.
PRETRAIN_MUON_LR = 0.02
PRETRAIN_ADAM_LR = 3e-4

# SSONN uses fixed per-cycle token schedules so every post-expansion iteration
# sees the same sequence windows and token order as the pre-expansion iteration.
SSONN_EVAL_BATCHES = 16
FINDER_BATCHES = 32
SSONN_LEARNING_RATE = 1e-6
SSONN_CYCLE_STEPS = 25_000
SSONN_EXPANSION_THRESHOLD = 0.1
SSONN_MAX_NEW_EDGES = 1024
REINIT_MODE = "full"  # Reinitialize NanoGPT and SSONN weights after expansion.

RUN_MODE_TO_MLP_KIND = {
    "pretrain": "none",
    "ssonn": "ssonn",
    "mlp": "speedrun_mlp",
}
MODE_ID = {"pretrain": 0, "ssonn": 1, "mlp": 2}
if RUN_MODE not in RUN_MODE_TO_MLP_KIND:
    raise ValueError(f"Unsupported RUN_MODE={RUN_MODE!r}; expected one of {tuple(RUN_MODE_TO_MLP_KIND)}")
MLP_KIND = RUN_MODE_TO_MLP_KIND[RUN_MODE]
INIT_MODE = "scratch"
FIXED_CYCLE_TOKENS = RUN_MODE == "ssonn"
FULL_REINIT_AFTER_EXPANSION = RUN_MODE == "ssonn"

BASE_CHECKPOINT = CHECKPOINT_DIR / "finewebedu_attention_only.pt"
SSONN_CHECKPOINT = CHECKPOINT_DIR / "finewebedu_ssonn_best.pt"
MLP_CHECKPOINT = CHECKPOINT_DIR / "finewebedu_speedrun_mlp_best.pt"

assert MODEL_CONFIG.n_embd % MODEL_CONFIG.n_head == 0
assert WINDOW_START <= WINDOW_END <= MODEL_CONFIG.max_seq_len
assert MICRO_BATCH_SIZE == 1, "The stable document stream uses one sequence per microbatch."
print("effective batch tokens at max window:", GRAD_ACCUM_STEPS * WINDOW_END)
print("run mode:", RUN_MODE, "| mlp kind:", MLP_KIND)

In [ ]:
experiment.end()

In [ ]:
experiment = None
if os.environ.get("COMET_API_KEY") and os.environ.get("COMET_WORKSPACE"):
    from comet_ml import start

    experiment = start(
        api_key=os.environ["COMET_API_KEY"],
        workspace=os.environ["COMET_WORKSPACE"],
        project_name=os.environ.get("COMET_PROJECT_NAME", "ssonn-experiments"),
    )
    experiment.set_name("ssonn small lr")
    experiment.log_parameters({
        "run_mode": RUN_MODE,
        "mlp_kind": MLP_KIND,
        "init_mode": INIT_MODE,
        "fixed_cycle_tokens": FIXED_CYCLE_TOKENS,
        "full_reinit_after_expansion": FULL_REINIT_AFTER_EXPANSION,
        "model": asdict(MODEL_CONFIG),
        "num_updates": NUM_UPDATES,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "window_start": WINDOW_START,
        "window_end": WINDOW_END,
        "cooldown_updates": COOLDOWN_UPDATES,
        "ssonn_learning_rate": SSONN_LEARNING_RATE,
        "ssonn_cycle_steps": SSONN_CYCLE_STEPS,
        "ssonn_expansion_threshold": SSONN_EXPANSION_THRESHOLD,
        "ssonn_max_new_edges": SSONN_MAX_NEW_EDGES,
        "source_recipe": "Deveraux-Parker/nanoGPT_1GPU_SPEEDRUN@135bc07555d47b64223c03e3da0bc6571efc6f2a",
        "stable_deviations": "no FlexAttention, FP8, BF16, torchao, or custom Triton kernels",
        "gpu": torch.cuda.get_device_name(),
    })
else:
    print("Comet logging disabled: set COMET_API_KEY and COMET_WORKSPACE to enable it.")

## FineWeb-Edu Data

The recommended path downloads the pretokenized GPT-2 BPE shards published for the speedrun. It avoids network streaming, parquet parsing, and local BPE tokenization.

Set `DOWNLOAD_PRETOKENIZED_DATA = True` once. The fallback local tokenizer remains available as `PREPARE_DATA = True`, but writes each shard incrementally rather than holding one billion Python integers in memory.

In [7]:
def write_datafile(path: Path, tokens: np.ndarray) -> None:
    header = np.zeros(256, dtype=np.int32)
    header[0] = 20240520
    header[1] = 1
    header[2] = len(tokens)
    with path.open("wb") as f:
        f.write(header.tobytes())
        f.write(tokens.astype(np.uint16, copy=False).tobytes())


def download_pretokenized_fineweb_edu(train_shards: int) -> None:
    from huggingface_hub import hf_hub_download

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    repo_id = "kjj0/finewebedu10B-gpt2"
    filenames = ["finewebedu_val_000000.bin"]
    filenames.extend(f"finewebedu_train_{index:06d}.bin" for index in range(1, train_shards + 1))
    for filename in filenames:
        print("Downloading", filename)
        hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filename, local_dir=DATA_DIR)


def prepare_fineweb_edu(train_token_limit: int, val_token_limit: int, shard_size: int = 100_000_000) -> None:
    from array import array
    import tiktoken

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    enc = tiktoken.get_encoding("gpt2")
    eot = enc.eot_token
    dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    rows = iter(dataset)

    def fill_buffer(limit: int) -> np.ndarray:
        buffer = array("H")
        for row in tqdm(rows, desc=f"Tokenizing {limit / 1e6:.0f}M tokens"):
            buffer.extend(enc.encode_ordinary(row["text"]))
            buffer.append(eot)
            if len(buffer) >= limit:
                return np.frombuffer(buffer, dtype=np.uint16, count=limit).copy()
        raise RuntimeError("FineWeb-Edu stream ended before reaching the requested token limit.")

    val_tokens = fill_buffer(val_token_limit)
    write_datafile(DATA_DIR / "fineweb_edu_val_000000.bin", val_tokens)

    written_tokens = 0
    shard_index = 0
    while written_tokens < train_token_limit:
        count = min(shard_size, train_token_limit - written_tokens)
        shard_tokens = fill_buffer(count)
        write_datafile(DATA_DIR / f"fineweb_edu_train_{shard_index:06d}.bin", shard_tokens)
        written_tokens += len(shard_tokens)
        shard_index += 1
        print(f"Wrote {written_tokens / 1e6:.0f}M / {train_token_limit / 1e6:.0f}M train tokens")


if DOWNLOAD_PRETOKENIZED_DATA:
    download_pretokenized_fineweb_edu(PRETOKENIZED_TRAIN_SHARDS)
elif PREPARE_DATA:
    prepare_fineweb_edu(PREPARE_TRAIN_TOKENS, PREPARE_VAL_TOKENS)

In [ ]:
def read_shard(path: str) -> np.ndarray:
    raw = np.memmap(path, dtype=np.uint16, mode="r")
    header = np.memmap(path, dtype=np.int32, mode="r", shape=(256,))
    if int(header[0]) != 20240520:
        raise ValueError(f"Unexpected FineWeb-Edu shard header in {path}")
    token_count = int(header[2])
    return raw[512 : 512 + token_count]


class TokenShardStream:
    def __init__(self, pattern: str, seed: int = 0):
        self.files = sorted(glob.glob(pattern))
        if not self.files:
            raise FileNotFoundError(f"No shards found for {pattern}. Run the preparation cell first.")
        self.rng = random.Random(seed)
        self.shard_index = 0
        self.position = 0
        self.tokens = read_shard(self.files[self.shard_index])

    def _advance(self) -> None:
        self.shard_index = (self.shard_index + 1) % len(self.files)
        self.position = 0
        self.tokens = read_shard(self.files[self.shard_index])

    def next_batch(self, seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
        required = seq_len + 1
        if self.position + required >= len(self.tokens):
            self._advance()
        block = torch.from_numpy(np.array(self.tokens[self.position : self.position + required], dtype=np.int64))
        self.position += seq_len
        return block[:-1].pin_memory(), block[1:].pin_memory()


class FixedFinderLoader:
    def __init__(self, pattern: str, batches: int, seq_len: int, seed: int):
        self.pattern = pattern
        self.batches = batches
        self.seq_len = seq_len
        self.seed = seed

    def __len__(self) -> int:
        return self.batches

    def __iter__(self):
        stream = TokenShardStream(self.pattern, self.seed)
        for _ in range(self.batches):
            x, y = stream.next_batch(self.seq_len)
            yield x.unsqueeze(0), y.unsqueeze(0)


def to_device(x: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    return x.unsqueeze(0).to(device, non_blocking=True), y.unsqueeze(0).to(device, non_blocking=True)

# Show the expected paths without stopping notebook setup before data is downloaded.
train_files = sorted(glob.glob(TRAIN_GLOB))
val_files = sorted(glob.glob(VAL_GLOB))
if train_files and val_files:
    train_stream = TokenShardStream(TRAIN_GLOB, seed=SEED)
    val_stream = TokenShardStream(VAL_GLOB, seed=SEED + 1)
    x0, y0 = train_stream.next_batch(WINDOW_START)
    print("data dir:", DATA_DIR)
    print("train shards:", len(train_files), "val shards:", len(val_files), "sample:", tuple(x0.shape), tuple(y0.shape))
else:
    print("No FineWeb-Edu shards found in", DATA_DIR)
    print("Set DOWNLOAD_PRETOKENIZED_DATA = True and run the previous cell once.")

## Stable Speedrun-Style Attention-Only GPT

Transferred from the speedrun: 12 layers, 6 heads, width 768, GPT-2 vocabulary, RoPE, RMSNorm/QK-norm, MQA parameterization, zero-initialized attention output projection, Muon for matrices, AdamW for embeddings/norm scalars, and tanh logit softcap.

Not transferred: FlexAttention/document block masks, FP8, BF16, `torchao`, and custom Triton kernels. Those require the nightly/Hopper environment explicitly excluded by this experiment.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, width: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps

    def reset_parameters(self) -> None:
        nn.init.ones_(self.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.rms_norm(x, (x.size(-1),), self.weight, self.eps)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, base: float = 10_000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, heads, seq, head_dim]
        seq_len = x.size(-2)
        cos = self.cos[:seq_len].to(dtype=x.dtype, device=x.device)[None, None]
        sin = self.sin[:seq_len].to(dtype=x.dtype, device=x.device)[None, None]
        x_even, x_odd = x[..., 0::2], x[..., 1::2]
        rotated = torch.stack((x_even * cos - x_odd * sin, x_even * sin + x_odd * cos), dim=-1)
        return rotated.flatten(-2)


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.q_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.k_proj = nn.Linear(config.n_embd, self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, config.max_seq_len)
        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.q_proj.weight, std=0.5 * self.n_embd ** -0.5)
        nn.init.normal_(self.k_proj.weight, std=0.5 * self.n_embd ** -0.5)
        nn.init.normal_(self.v_proj.weight, std=0.5 * self.n_embd ** -0.5)
        nn.init.zeros_(self.o_proj.weight)
        self.q_norm.reset_parameters()
        self.k_norm.reset_parameters()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch, seq_len, 1, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch, seq_len, 1, self.head_dim).transpose(1, 2)
        q = self.rope(self.q_norm(q))
        k = self.rope(self.k_norm(k))
        # Expand one KV head instead of requiring nightly FlexAttention/GQA kernels.
        k = k.expand(-1, self.n_head, -1, -1)
        v = v.expand(-1, self.n_head, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(batch, seq_len, -1))


class TokenSSONNMLP(nn.Module):
    def __init__(self, width: int, device_name: str):
        super().__init__()
        # Start as a dense Kaiming-uniform projection, like later SSONN resets.
        # Ones makes the initial sparsity mask dense before reinitialization.
        self.scaled = ScaledNet(torch.ones((width, width)), torch.zeros(width), device=device_name)
        self.scaled.dropout.p = 0.0
        self.reset_parameters()

    def reset_parameters(self) -> None:
        self.scaled.reinitialize_parameters()
        self.scaled.dropout.p = 0.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, width = x.shape
        return self.scaled(x.reshape(batch * seq_len, width)).reshape(batch, seq_len, width)


class SpeedrunMLP(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.width = width
        self.c_fc = nn.Linear(width, 4 * width, bias=False)
        self.c_proj = nn.Linear(4 * width, width, bias=False)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.normal_(self.c_fc.weight, std=0.5 * self.width ** -0.5)
        nn.init.zeros_(self.c_proj.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.c_proj(F.relu(self.c_fc(x)).square())


class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig, mlp_kind: str = "none"):
        super().__init__()
        if mlp_kind not in {"none", "ssonn", "speedrun_mlp"}:
            raise ValueError(f"Unsupported mlp_kind={mlp_kind!r}")
        self.mlp_kind = mlp_kind
        self.norm_attn = RMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.norm_mlp = RMSNorm(config.n_embd)
        if mlp_kind == "ssonn":
            self.mlp = TokenSSONNMLP(config.n_embd, str(device))
        elif mlp_kind == "speedrun_mlp":
            self.mlp = SpeedrunMLP(config.n_embd)
        else:
            self.mlp = None

    def reset_parameters(self) -> None:
        self.norm_attn.reset_parameters()
        self.attn.reset_parameters()
        self.norm_mlp.reset_parameters()
        if self.mlp is not None and hasattr(self.mlp, "reset_parameters"):
            self.mlp.reset_parameters()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm_attn(x))
        if self.mlp_kind == "ssonn":
            x = x + self.mlp(self.norm_mlp(x))
        elif self.mlp_kind == "speedrun_mlp":
            x = x + self.mlp(self.norm_mlp(x))
        return x


class AttentionOnlyGPT(nn.Module):
    def __init__(self, config: GPTConfig, mlp_kind: str = "none"):
        super().__init__()
        self.config = config
        self.mlp_kind = mlp_kind
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(config, mlp_kind) for _ in range(config.n_layer)])
        self.norm_final = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.ssonn_layer_names: list[str] = []
        if mlp_kind == "ssonn":
            for idx, block in enumerate(self.blocks):
                name = f"ssonn_mlp_{idx}"
                setattr(self, name, block.mlp.scaled)
                self.ssonn_layer_names.append(name)

    def reset_parameters(self) -> None:
        self.wte.reset_parameters()
        for block in self.blocks:
            block.reset_parameters()
        self.norm_final.reset_parameters()
        self.lm_head.weight = self.wte.weight

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        x = self.wte(tokens)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm_final(x))
        return 30.0 * torch.tanh(logits / 30.0)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


def count_nonzero_parameters(model: nn.Module) -> int:
    return sum((parameter.detach() != 0).sum().item() for parameter in model.parameters())


def count_ssonn_sublinears(model: nn.Module) -> int:
    return sum(len(module.sub_linears) for module in model.modules() if isinstance(module, ScaledNet))


model = AttentionOnlyGPT(MODEL_CONFIG, mlp_kind=MLP_KIND).to(device)
if USE_COMPILE:
    model = torch.compile(model)
print(f"parameters: {count_parameters(model) / 1e6:.1f}M | mlp kind: {MLP_KIND} | ssonn layers: {len(model.ssonn_layer_names)}")

## Checkpoint Interface

SSONN and dense MLP comparison modes start from scratch. Checkpoint loading is kept only for attention-only pretrain/evaluation helpers.

In [10]:
def checkpoint_payload(model: nn.Module, optimizer_state: dict | None, update: int) -> dict:
    return {
        "model": model.state_dict(),
        "model_config": asdict(MODEL_CONFIG),
        "update": update,
        "optimizer": optimizer_state,
        "recipe": {
            "source": "Deveraux-Parker/nanoGPT_1GPU_SPEEDRUN@135bc07555d47b64223c03e3da0bc6571efc6f2a",
            "stable_deviations": ["no FlexAttention", "no FP8", "no BF16", "no torchao", "standard SDPA"],
            "num_updates": NUM_UPDATES,
            "window_start": WINDOW_START,
            "window_end": WINDOW_END,
            "effective_sequences_per_update": GRAD_ACCUM_STEPS,
        },
        "rng": {
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all(),
            "numpy": np.random.get_state(),
            "python": random.getstate(),
        },
    }


def assert_finite_model(model: nn.Module, label: str) -> None:
    for name, parameter in model.named_parameters():
        if not torch.isfinite(parameter).all():
            raise FloatingPointError(f"{label}: non-finite parameter '{name}'.")


def assert_finite_tensor(value: torch.Tensor, label: str) -> None:
    if not torch.isfinite(value).all():
        raise FloatingPointError(f"{label}: non-finite value.")


def save_checkpoint(path: Path, model: nn.Module, optimizer_state: dict | None, update: int) -> None:
    assert_finite_model(model, f"Refusing to save {path.name}")
    torch.save(checkpoint_payload(model, optimizer_state, update), path)
    print("saved", path)


def load_attention_checkpoint(model: AttentionOnlyGPT, path: Path) -> dict:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    missing, unexpected = model.load_state_dict(checkpoint["model"], strict=False)
    if unexpected:
        raise RuntimeError(f"Unexpected checkpoint keys: {unexpected}")
    if not all("mlp" in name or "ssonn_mlp" in name for name in missing):
        raise RuntimeError(f"Checkpoint is incompatible with this backbone. Missing: {missing}")
    assert_finite_model(model, f"Checkpoint {path}")
    return checkpoint


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## Speedrun-Inspired Pretraining

Matrix parameters use Muon; embeddings and norm scales use AdamW. This is the stable counterpart to the source's Muon/Adam split. The window schedule and cooldown are update-based, exactly as in the source recipe. `GRAD_ACCUM_STEPS=16` preserves its effective sequence count while keeping a 24GB TITAN RTX safe with standard attention.

In [11]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    stream: TokenShardStream,
    seq_len: int,
    batches: int,
    use_amp: bool = True,
) -> float:
    model.eval()
    losses = []
    for _ in range(batches):
        x_cpu, y_cpu = stream.next_batch(seq_len)
        x, y = to_device(x_cpu, y_cpu)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=use_amp):
            logits = model(x)
            assert_finite_tensor(logits, "Evaluation logits")
            loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
        assert_finite_tensor(loss, "Evaluation loss")
        losses.append(loss.item())
    return float(np.mean(losses))


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr: float = 0.02, momentum: float = 0.95, nesterov: bool = True, steps: int = 5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, steps=steps))

    @staticmethod
    def orthogonalize(gradient: torch.Tensor, steps: int) -> torch.Tensor:
        x = gradient.float()
        x = x / (x.norm() + 1e-7)
        transposed = x.size(0) > x.size(1)
        if transposed:
            x = x.t()
        a, b, c = 3.4445, -4.7750, 2.0315
        for _ in range(steps):
            a_mat = x @ x.t()
            x = a * x + (b * a_mat + c * a_mat @ a_mat) @ x
        if transposed:
            x = x.t()
        return x.to(dtype=gradient.dtype)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue
                state = self.state[parameter]
                if "momentum" not in state:
                    state["momentum"] = torch.zeros_like(parameter.grad)
                momentum = state["momentum"]
                momentum.mul_(group["momentum"]).add_(parameter.grad)
                update = parameter.grad.add(momentum, alpha=group["momentum"]) if group["nesterov"] else momentum
                update = self.orthogonalize(update, group["steps"])
                update.mul_(max(1.0, update.size(0) / update.size(1)) ** 0.5)
                parameter.add_(update, alpha=-group["lr"])


def make_pretrain_optimizers(model: nn.Module):
    matrix, adam = [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if parameter.ndim == 2 and name not in {"wte.weight", "lm_head.weight"}:
            matrix.append(parameter)
        else:
            adam.append(parameter)
    return (
        Muon(matrix, lr=PRETRAIN_MUON_LR, momentum=0.85),
        torch.optim.AdamW(adam, lr=PRETRAIN_ADAM_LR, betas=(0.8, 0.95), weight_decay=0.0),
    )


def speedrun_window(step: int) -> int:
    fraction = step / max(NUM_UPDATES - 1, 1)
    raw = WINDOW_START + fraction * (WINDOW_END - WINDOW_START)
    return min(WINDOW_END, max(WINDOW_START, int(raw // 256) * 256))


def speedrun_lr_scale(step: int) -> float:
    cooldown_start = NUM_UPDATES - COOLDOWN_UPDATES
    return 1.0 if step < cooldown_start else (NUM_UPDATES - step) / COOLDOWN_UPDATES


def pretrain_attention_only(model: AttentionOnlyGPT) -> AttentionOnlyGPT:
    if RUN_MODE != "pretrain":
        raise ValueError("Set RUN_MODE='pretrain' for this cell.")
    model.train()
    muon, adam = make_pretrain_optimizers(model)
    scaler = torch.amp.GradScaler("cuda")
    train_stream = TokenShardStream(TRAIN_GLOB, seed=SEED)
    val_stream = TokenShardStream(VAL_GLOB, seed=SEED + 1)
    started = time.perf_counter()
    tokens_seen = 0

    for step in range(NUM_UPDATES):
        seq_len = speedrun_window(step)
        muon.param_groups[0]["momentum"] = 0.85 + 0.10 * min(step / 300, 1.0)
        lr_scale = speedrun_lr_scale(step)
        muon.param_groups[0]["lr"] = PRETRAIN_MUON_LR * lr_scale
        adam.param_groups[0]["lr"] = PRETRAIN_ADAM_LR * lr_scale
        muon.zero_grad(set_to_none=True)
        adam.zero_grad(set_to_none=True)

        loss_sum = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            x_cpu, y_cpu = train_stream.next_batch(seq_len)
            x, y = to_device(x_cpu, y_cpu)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten()) / GRAD_ACCUM_STEPS
            assert_finite_tensor(loss, f"Pretrain loss at update {step + 1}")
            scaler.scale(loss).backward()
            # loss is already divided by GRAD_ACCUM_STEPS; summing it gives the mean raw loss.
            loss_sum += loss.detach().item()

        tokens_seen += GRAD_ACCUM_STEPS * seq_len
        scaler.unscale_(muon)
        scaler.unscale_(adam)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        grad_norm_finite = torch.isfinite(grad_norm)
        skipped_step = not bool(grad_norm_finite.item())
        if skipped_step:
            scaler.update()
            muon.zero_grad(set_to_none=True)
            adam.zero_grad(set_to_none=True)
            print(f"update {step + 1:4d}/{NUM_UPDATES} | skipped non-finite gradients | scale {scaler.get_scale():.1f}")
            if experiment is not None:
                experiment.log_metrics({
                    "ssonn/skipped_step": 1,
                    "ssonn/grad_norm_finite": 0,
                    "ssonn/grad_scale": scaler.get_scale(),
                    "ssonn/window": seq_len,
                    "ssonn/tokens_seen": tokens_seen,
                }, step=step + 1)
            continue
        scaler.step(muon)
        scaler.step(adam)
        scaler.update()
        assert_finite_model(model, f"Pretrain model after update {step + 1}")

        if (step + 1) % EVAL_INTERVAL == 0 or step == 0:
            val_loss = evaluate(model, val_stream, seq_len, EVAL_BATCHES)
            elapsed = time.perf_counter() - started
            tokens_per_second = tokens_seen / max(elapsed, 1)
            print(f"update {step + 1:4d}/{NUM_UPDATES} | window {seq_len:4d} | train {loss_sum:.4f} | val {val_loss:.4f} | {tokens_per_second:.0f} tok/s")
            if experiment is not None:
                experiment.log_metrics({
                    "ssonn/train_loss": loss_sum,
                    "ssonn/val_loss": val_loss,
                    "ssonn/val_perplexity": math.exp(min(val_loss, 20.0)),
                    "ssonn/window": seq_len,
                    "ssonn/tokens_seen": tokens_seen,
                    "ssonn/tokens_per_second": tokens_per_second,
                    "ssonn/elapsed_seconds": elapsed,
                    "ssonn/grad_norm": float(grad_norm),
                    "ssonn/grad_norm_finite": 1,
                    "ssonn/skipped_step": 0,
                    "ssonn/grad_scale": scaler.get_scale(),
                    "ssonn/muon_lr": muon.param_groups[0]["lr"],
                    "ssonn/adam_lr": adam.param_groups[0]["lr"],
                }, step=step + 1)
            model.train()

        if (step + 1) % CHECKPOINT_INTERVAL == 0 or step + 1 == NUM_UPDATES:
            save_checkpoint(
                BASE_CHECKPOINT,
                model,
                {"muon": muon.state_dict(), "adam": adam.state_dict()},
                step + 1,
            )
    return model


# Run this only after FineWeb-Edu shards exist.
# pretrained_model = pretrain_attention_only(model)

In [ ]:
def summarize_prior_logs(log_root: Path = REPO_ROOT / "logs") -> None:
    if not log_root.exists():
        print("No prior logs found at", log_root)
        return

    def chart_series(path: Path) -> list[tuple[int | float, float]]:
        data = json.loads(path.read_text())
        if not data:
            return []
        xs = data[0].get("x", [])
        ys = data[0].get("y", [])
        return [
            (x, float(y))
            for x, y in zip(xs, ys)
            if isinstance(y, (int, float)) and math.isfinite(y)
        ]

    def print_loss_summary(label: str, path: Path) -> None:
        points = chart_series(path)
        if not points:
            return
        best_step, best_value = min(points, key=lambda pair: pair[1])
        last_step, last_value = points[-1]
        print(f"{label}: best {best_value:.4f} @ {best_step}, last {last_value:.4f} @ {last_step}")

    def print_jump_summary(label: str, path: Path) -> None:
        points = chart_series(path)
        jumps = [
            (y1 - y0, x0, x1, y0, y1)
            for (x0, y0), (x1, y1) in zip(points, points[1:])
            if y1 > y0
        ]
        if not jumps:
            return
        delta, x0, x1, y0, y1 = max(jumps, key=lambda item: item[0])
        print(f"{label}: largest upward jump +{delta:.4f}: {y0:.4f} @ {x0} -> {y1:.4f} @ {x1}")

    print("Prior log summary")
    for path in sorted(log_root.rglob("*pretrain_train_loss*chart_data.json")):
        print_loss_summary("pretrain/train_loss", path)
    for path in sorted(log_root.rglob("*ssonn_val_loss*chart_data.json")):
        print_loss_summary("ssonn/val_loss", path)
        print_jump_summary("ssonn/val_loss", path)
    for path in sorted(log_root.rglob("*ssonn_grad_norm*chart_data.json")):
        points = chart_series(path)
        if points:
            max_step, max_value = max(points, key=lambda pair: pair[1])
            print(f"ssonn/grad_norm: max {max_value:.4f} @ {max_step}")
            print_jump_summary("ssonn/grad_norm", path)
    for path in sorted(log_root.rglob("*metrics.csv")):
        rows = {}
        with path.open() as file:
            header = file.readline().strip().split(",")
            for line in file:
                values = line.strip().split(",")
                if len(values) == len(header):
                    rows[values[0]] = dict(zip(header, values))
        for key in ("ssonn/base_val_loss", "ssonn/initial_adapter_val_loss", "ssonn/expanded", "ssonn/new_edges", "ssonn/total_params"):
            if key in rows:
                row = rows[key]
                print(f"{key}: last {row['LAST_VALUE']} @ {row['LAST_STEP']}, min {row['MIN_VALUE']} @ {row['MIN_STEP']}, max {row['MAX_VALUE']} @ {row['MAX_STEP']}")


summarize_prior_logs()

## SSONN And MLP Comparison Stages

Switch to `RUN_MODE = "ssonn"` for from-scratch NanoGPT+SSONN or `RUN_MODE = "mlp"` for from-scratch NanoGPT+speedrun-MLP. The attention-only `pretrain` path is intentionally left unchanged. SSONN cycles use local token schedules: after expansion, the train stream and window curriculum reset so each SSONN iteration sees the same token budget and order.



In [ ]:
def ssonn_only_weights_init(module: nn.Module) -> None:
    if isinstance(module, ScaledNet):
        module.reinitialize_parameters()


def full_model_reinit(model: AttentionOnlyGPT) -> None:
    model.reset_parameters()


def lm_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # EdgeFinder calls metric.calculate(...), which applies squeeze() to a batch of one.
    # Support both regular [batch, seq, vocab] logits and squeezed [seq, vocab] logits.
    if logits.ndim == 3:
        logits = logits.flatten(0, 1)
    elif logits.ndim != 2:
        raise ValueError(f"Expected [batch, seq, vocab] or [seq, vocab], got {tuple(logits.shape)}")
    return F.cross_entropy(logits, targets.flatten())


def ssonn_parameter_ids(model: nn.Module) -> set[int]:
    return {
        id(parameter)
        for module in model.modules()
        if isinstance(module, ScaledNet)
        for parameter in module.parameters()
    }


def make_ssonn_stage_optimizers(model: nn.Module):
    ssonn_ids = ssonn_parameter_ids(model)
    muon_parameters, backbone_adam_parameters, ssonn_parameters = [], [], []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if id(parameter) in ssonn_ids:
            ssonn_parameters.append(parameter)
        elif parameter.ndim == 2 and name not in {"wte.weight", "lm_head.weight"}:
            muon_parameters.append(parameter)
        else:
            backbone_adam_parameters.append(parameter)

    return (
        Muon(muon_parameters, lr=PRETRAIN_MUON_LR, momentum=0.85),
        torch.optim.AdamW(backbone_adam_parameters, lr=PRETRAIN_ADAM_LR, betas=(0.8, 0.95), weight_decay=0.0),
        torch.optim.AdamW(ssonn_parameters, lr=0.0, weight_decay=0.0),
    )


def ssonn_cosine_lr(cycle_step: int) -> float:
    if SSONN_CYCLE_STEPS <= 0:
        raise ValueError("SSONN_CYCLE_STEPS must be positive.")
    denominator = max(SSONN_CYCLE_STEPS - 1, 1)
    position = min(max(cycle_step, 0), denominator)
    return 0.5 * SSONN_LEARNING_RATE * (1.0 - math.cos(2.0 * math.pi * position / denominator))


def cycle_window(cycle_step: int) -> int:
    denominator = max(SSONN_CYCLE_STEPS - 1, 1)
    fraction = min(max(cycle_step, 0), denominator) / denominator
    raw = WINDOW_START + fraction * (WINDOW_END - WINDOW_START)
    return min(WINDOW_END, max(WINDOW_START, int(raw // 256) * 256))


def layer_input_widths(model: AttentionOnlyGPT) -> dict[str, int]:
    return {name: getattr(model, name).in_features for name in model.ssonn_layer_names}


def make_edge_finder(seq_len: int) -> EdgeFinder:
    return EdgeFinder(
        metric=AbsGradientEdgeMetric(lm_cross_entropy),
        dataloader=FixedFinderLoader(TRAIN_GLOB, FINDER_BATCHES, seq_len, SEED + 2),
        device=device,
        aggregation_mode="mean",
        max_to_replace=SSONN_MAX_NEW_EDGES,
    )


def comparison_metrics(
    mode: str,
    train_loss: float,
    val_loss: float,
    seq_len: int,
    cycle_step: int,
    cycle_tokens_seen: int,
    global_tokens_seen: int,
    tokens_per_second: float,
    grad_norm: torch.Tensor,
    elapsed: float,
) -> dict[str, float]:
    return {
        "compare/mode": MODE_ID[mode],
        "compare/train_loss": train_loss,
        "compare/val_loss": val_loss,
        "compare/val_perplexity": math.exp(min(val_loss, 20.0)),
        "compare/window": seq_len,
        "compare/cycle_step": cycle_step,
        "compare/cycle_tokens_seen": cycle_tokens_seen,
        "compare/global_tokens_seen": global_tokens_seen,
        "compare/tokens_per_second": tokens_per_second,
        "compare/grad_norm": float(grad_norm),
        "compare/elapsed_seconds": elapsed,
    }


def train_ssonn_stage(model: AttentionOnlyGPT) -> AttentionOnlyGPT:
    if RUN_MODE != "ssonn":
        raise ValueError("Set RUN_MODE='ssonn' and rerun model construction before this cell.")

    full_model_reinit(model)
    ssonn_train.weights_init = ssonn_only_weights_init

    muon, backbone_adam, ssonn_adam = make_ssonn_stage_optimizers(model)
    ssonn_train.apply_sparsity_mask(model)
    thresholds = {name: SSONN_EXPANSION_THRESHOLD for name in model.ssonn_layer_names}
    train_stream = TokenShardStream(TRAIN_GLOB, SEED + 3)
    val_stream = TokenShardStream(VAL_GLOB, SEED + 4)
    initial_loss = evaluate(model, TokenShardStream(VAL_GLOB, SEED + 4), WINDOW_START, SSONN_EVAL_BATCHES, use_amp=False)
    print(f"from-scratch full SSONN initial val loss: {initial_loss:.4f}")
    best_val = float("inf")
    global_tokens_seen = 0
    cycle_tokens_seen = 0
    cycle_index = 0
    cycle_start = 0
    started = time.perf_counter()

    for step in range(NUM_UPDATES):
        cycle_step = step - cycle_start
        seq_len = cycle_window(cycle_step)
        lr_scale = speedrun_lr_scale(cycle_step)
        muon.param_groups[0]["momentum"] = 0.85 + 0.10 * min(cycle_step / 300, 1.0)
        muon.param_groups[0]["lr"] = PRETRAIN_MUON_LR * lr_scale
        backbone_adam.param_groups[0]["lr"] = PRETRAIN_ADAM_LR * lr_scale
        ssonn_lr = ssonn_cosine_lr(cycle_step)
        ssonn_adam.param_groups[0]["lr"] = ssonn_lr

        model.train()
        for optimizer in (muon, backbone_adam, ssonn_adam):
            optimizer.zero_grad(set_to_none=True)

        train_loss = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            x_cpu, y_cpu = train_stream.next_batch(seq_len)
            x, y = to_device(x_cpu, y_cpu)
            # Full SSONN adapters can still overflow FP16 backward before clipping.
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=False):
                loss = lm_cross_entropy(model(x), y)
            assert_finite_tensor(loss, f"SSONN loss at update {step + 1}")
            (loss / GRAD_ACCUM_STEPS).backward()
            train_loss += loss.detach().item() / GRAD_ACCUM_STEPS

        step_tokens = GRAD_ACCUM_STEPS * seq_len
        global_tokens_seen += step_tokens
        cycle_tokens_seen += step_tokens
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        assert_finite_tensor(grad_norm, f"SSONN gradient norm at update {step + 1}")
        for optimizer in (muon, backbone_adam, ssonn_adam):
            optimizer.step()
        assert_finite_model(model, f"SSONN model after update {step + 1}")

        expanded = False
        new_edges = 0
        full_reinit = False
        pre_expansion_val_loss = float("nan")
        post_reinit_val_loss = float("nan")
        new_edges_by_layer = {name: 0 for name in model.ssonn_layer_names}
        log_cycle_index = cycle_index
        log_cycle_step = cycle_step
        log_cycle_tokens_seen = cycle_tokens_seen

        cycle_finished = cycle_step + 1 >= SSONN_CYCLE_STEPS
        if cycle_finished:
            expansion_val_stream = TokenShardStream(VAL_GLOB, SEED + 4)
            pre_expansion_val_loss = evaluate(model, expansion_val_stream, seq_len, SSONN_EVAL_BATCHES, use_amp=False)
            before_widths = layer_input_widths(model)
            expanded, new_edges = ssonn_train.expand_model(model, make_edge_finder(seq_len), thresholds)
            after_widths = layer_input_widths(model)
            new_edges_by_layer = {
                name: after_widths[name] - before_widths[name]
                for name in model.ssonn_layer_names
            }
            if expanded:
                full_model_reinit(model)
                full_reinit = True
                muon, backbone_adam, ssonn_adam = make_ssonn_stage_optimizers(model)
                ssonn_train.apply_sparsity_mask(model)
                train_stream = TokenShardStream(TRAIN_GLOB, seed=SEED + 3)
                post_reinit_val_loss = evaluate(model, TokenShardStream(VAL_GLOB, SEED + 4), WINDOW_START, SSONN_EVAL_BATCHES, use_amp=False)
                assert_finite_model(model, f"SSONN model after full reinit at update {step + 1}")
            cycle_index += 1
            cycle_start = step + 1
            cycle_tokens_seen = 0

        should_eval = (step + 1) % EVAL_INTERVAL == 0 or step == 0 or expanded or step + 1 == NUM_UPDATES
        if should_eval:
            val_loss = evaluate(model, val_stream, seq_len, SSONN_EVAL_BATCHES, use_amp=False)
            elapsed = time.perf_counter() - started
            tokens_per_second = global_tokens_seen / max(elapsed, 1.0)
            if val_loss >= 20.0:
                ppl_text = ">= exp(20)"
            else:
                ppl_text = f"{math.exp(val_loss):.2f}"
            total_params = count_parameters(model)
            nonzero_params = count_nonzero_parameters(model)
            print(
                f"update {step + 1:4d}/{NUM_UPDATES} | cycle {log_cycle_index:02d} step {log_cycle_step + 1:5d}/{SSONN_CYCLE_STEPS} | "
                f"window {seq_len:4d} | train {train_loss:.4f} | val {val_loss:.4f} | ppl {ppl_text} | "
                f"ssonn_lr {ssonn_lr:.2e} | expanded {expanded} edges {new_edges} | {tokens_per_second:.0f} tok/s"
            )
            if experiment is not None:
                metrics = {
                    "ssonn/train_loss": train_loss,
                    "ssonn/val_loss": val_loss,
                    "ssonn/val_perplexity": math.exp(min(val_loss, 20.0)),
                    "ssonn/window": seq_len,
                    "ssonn/cycle_index": log_cycle_index,
                    "ssonn/cycle_step": log_cycle_step,
                    "ssonn/cycle_tokens_seen": log_cycle_tokens_seen,
                    "ssonn/global_tokens_seen": global_tokens_seen,
                    "ssonn/tokens_per_second": tokens_per_second,
                    "ssonn/elapsed_seconds": elapsed,
                    "ssonn/grad_norm": float(grad_norm),
                    "ssonn/muon_lr": muon.param_groups[0]["lr"],
                    "ssonn/backbone_adam_lr": backbone_adam.param_groups[0]["lr"],
                    "ssonn/lr": ssonn_lr,
                    "ssonn/expanded": int(expanded),
                    "ssonn/new_edges": int(new_edges),
                    "ssonn/full_reinit": int(full_reinit),
                    "ssonn/total_params": total_params,
                    "ssonn/nonzero_params": nonzero_params,
                    "ssonn/sparsity": 1.0 - (nonzero_params / max(total_params, 1)),
                    "ssonn/sub_linear_count": count_ssonn_sublinears(model),
                    "ssonn/pre_expansion_val_loss": pre_expansion_val_loss,
                    "ssonn/post_reinit_val_loss": post_reinit_val_loss,
                }
                metrics.update(comparison_metrics(
                    "ssonn",
                    train_loss,
                    val_loss,
                    seq_len,
                    log_cycle_step,
                    log_cycle_tokens_seen,
                    global_tokens_seen,
                    tokens_per_second,
                    grad_norm,
                    elapsed,
                ))
                metrics.update({f"ssonn/new_edges/{name}": int(value) for name, value in new_edges_by_layer.items()})
                experiment.log_metrics(metrics, step=step + 1)
            model.train()

            if val_loss < best_val:
                best_val = val_loss
                save_checkpoint(
                    SSONN_CHECKPOINT,
                    model,
                    {
                        "muon": muon.state_dict(),
                        "backbone_adam": backbone_adam.state_dict(),
                        "ssonn_adam": ssonn_adam.state_dict(),
                    },
                    step + 1,
                )

        if (step + 1) % CHECKPOINT_INTERVAL == 0 or step + 1 == NUM_UPDATES:
            save_checkpoint(
                SSONN_CHECKPOINT.with_name("finewebedu_ssonn_latest.pt"),
                model,
                {
                    "muon": muon.state_dict(),
                    "backbone_adam": backbone_adam.state_dict(),
                    "ssonn_adam": ssonn_adam.state_dict(),
                },
                step + 1,
            )
    return model


def train_mlp_stage(model: AttentionOnlyGPT) -> AttentionOnlyGPT:
    if RUN_MODE != "mlp":
        raise ValueError("Set RUN_MODE='mlp' and rerun model construction before this cell.")
    model.train()
    muon, adam = make_pretrain_optimizers(model)
    scaler = torch.amp.GradScaler("cuda")
    train_stream = TokenShardStream(TRAIN_GLOB, seed=SEED + 3)
    val_stream = TokenShardStream(VAL_GLOB, seed=SEED + 4)
    started = time.perf_counter()
    tokens_seen = 0
    best_val = float("inf")

    for step in range(NUM_UPDATES):
        seq_len = speedrun_window(step)
        muon.param_groups[0]["momentum"] = 0.85 + 0.10 * min(step / 300, 1.0)
        lr_scale = speedrun_lr_scale(step)
        muon.param_groups[0]["lr"] = PRETRAIN_MUON_LR * lr_scale
        adam.param_groups[0]["lr"] = PRETRAIN_ADAM_LR * lr_scale
        muon.zero_grad(set_to_none=True)
        adam.zero_grad(set_to_none=True)

        train_loss = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            x_cpu, y_cpu = train_stream.next_batch(seq_len)
            x, y = to_device(x_cpu, y_cpu)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = lm_cross_entropy(model(x), y) / GRAD_ACCUM_STEPS
            assert_finite_tensor(loss, f"MLP loss at update {step + 1}")
            scaler.scale(loss).backward()
            train_loss += loss.detach().item()

        tokens_seen += GRAD_ACCUM_STEPS * seq_len
        scaler.unscale_(muon)
        scaler.unscale_(adam)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        grad_norm_finite = torch.isfinite(grad_norm)
        skipped_step = not bool(grad_norm_finite.item())
        if skipped_step:
            scaler.update()
            muon.zero_grad(set_to_none=True)
            adam.zero_grad(set_to_none=True)
            print(f"update {step + 1:4d}/{NUM_UPDATES} | skipped non-finite MLP gradients | scale {scaler.get_scale():.1f}")
            if experiment is not None:
                experiment.log_metrics({
                    "ssonn/skipped_step": 1,
                    "ssonn/grad_norm_finite": 0,
                    "ssonn/grad_scale": scaler.get_scale(),
                    "ssonn/window": seq_len,
                    "ssonn/global_tokens_seen": tokens_seen,
                }, step=step + 1)
            continue
        scaler.step(muon)
        scaler.step(adam)
        scaler.update()
        assert_finite_model(model, f"MLP model after update {step + 1}")

        if (step + 1) % EVAL_INTERVAL == 0 or step == 0 or step + 1 == NUM_UPDATES:
            val_loss = evaluate(model, val_stream, seq_len, EVAL_BATCHES)
            elapsed = time.perf_counter() - started
            tokens_per_second = tokens_seen / max(elapsed, 1.0)
            print(f"update {step + 1:4d}/{NUM_UPDATES} | window {seq_len:4d} | train {train_loss:.4f} | val {val_loss:.4f} | {tokens_per_second:.0f} tok/s")
            if experiment is not None:
                metrics = {
                    "ssonn/train_loss": train_loss,
                    "ssonn/val_loss": val_loss,
                    "ssonn/val_perplexity": math.exp(min(val_loss, 20.0)),
                    "ssonn/window": seq_len,
                    "ssonn/global_tokens_seen": tokens_seen,
                    "ssonn/tokens_per_second": tokens_per_second,
                    "ssonn/elapsed_seconds": elapsed,
                    "ssonn/grad_norm": float(grad_norm),
                    "ssonn/grad_norm_finite": 1,
                    "ssonn/skipped_step": 0,
                    "ssonn/grad_scale": scaler.get_scale(),
                    "ssonn/muon_lr": muon.param_groups[0]["lr"],
                    "ssonn/adam_lr": adam.param_groups[0]["lr"],
                }
                metrics.update(comparison_metrics(
                    "mlp",
                    train_loss,
                    val_loss,
                    seq_len,
                    step,
                    tokens_seen,
                    tokens_seen,
                    tokens_per_second,
                    grad_norm,
                    elapsed,
                ))
                experiment.log_metrics(metrics, step=step + 1)
            model.train()

            if val_loss < best_val:
                best_val = val_loss
                save_checkpoint(MLP_CHECKPOINT, model, {"muon": muon.state_dict(), "adam": adam.state_dict()}, step + 1)

        if (step + 1) % CHECKPOINT_INTERVAL == 0 or step + 1 == NUM_UPDATES:
            save_checkpoint(
                MLP_CHECKPOINT.with_name("finewebedu_speedrun_mlp_latest.pt"),
                model,
                {"muon": muon.state_dict(), "adam": adam.state_dict()},
                step + 1,
            )
    return model


# Run exactly one of these after setting RUN_MODE and rerunning model construction.
# ssonn_model = train_ssonn_stage(model)
# mlp_model = train_mlp_stage(model)